# DinoV2 Dogs-vs-Cats Training (Parallel 2xT4 DataParallel)

Simplified DinoV2 training notebook for Kaggle Dogs-vs-Cats.

Flow: bootstrap -> config -> preprocess -> sanity -> train -> review artifacts.


In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True,max_split_size_mb:128')
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True,max_split_size_mb:128')

REPO_URL = 'https://github.com/mruniverse8/kaggle-experiments-.git'
REPO_DIR = Path('/kaggle/working/kaggle-experiments-')
BRANCH = 'dogs_vs_cats_v2'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run(['git', 'fetch', '--all'], check=True)
subprocess.run(['git', 'checkout', BRANCH], check=True)
subprocess.run(['git', 'reset', '--hard', f'origin/{BRANCH}'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

current_branch = subprocess.check_output(['git', 'rev-parse', '--abbrev-ref', 'HEAD']).decode().strip()
current_commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD']).decode().strip()
print('Git branch:', current_branch)
print('Git commit:', current_commit)
print('Repo ready at:', REPO_DIR)



In [ ]:
import os
import json
from pathlib import Path

CFG_PATH = os.environ.get('CFG_PATH', 'dogs_vs_cats/configs/experiments/dinov2_vitb14_parallel_t4x2_small.json')
PATHS_CFG = os.environ.get('PATHS_CFG', 'dogs_vs_cats/configs/paths_kaggle.json')

CFG = json.loads(Path(CFG_PATH).read_text())
PATHS = json.loads(Path(PATHS_CFG).read_text())

print('Using experiment config:', CFG_PATH)
print('Using paths config:', PATHS_CFG)
print('Experiment name:', CFG['experiment_name'])
print('Batch train/eval:', CFG['data']['batch_size_train'], CFG['data']['batch_size_eval'])
print('Use data parallel:', CFG.get('use_data_parallel', False))
print('GPU ids:', CFG.get('parallel_gpu_ids', []))

for key in ['train_dir', 'eval_dir', 'test_dir', 'train_zip', 'test_zip', 'sample_submission_csv']:
    value = PATHS.get(key, '')
    if not value:
        print(f"{key}: <empty>")
        continue
    exists = Path(value).exists()
    print(f"{key}: {value} | exists={exists}")



In [ ]:
import sys
import subprocess
from pathlib import Path

manifest_dir = Path(PATHS['manifests_dir'])
required = [
    manifest_dir / 'train_manifest.csv',
    manifest_dir / 'val_manifest.csv',
    manifest_dir / 'test_manifest.csv',
]

if all(path.exists() for path in required):
    print('Preprocess skipped: manifests already exist')
else:
    subprocess.run([
        sys.executable,
        'dogs_vs_cats/src/preprocess_competition_data.py',
        '--paths-config', PATHS_CFG,
        '--experiment-config', CFG_PATH,
    ], check=True)



In [ ]:
import sys
import subprocess

subprocess.run([
    sys.executable,
    'dogs_vs_cats/src/sanity_check_random_init.py',
    '--paths-config', PATHS_CFG,
    '--experiment-config', CFG_PATH,
], check=True)


In [ ]:

import sys
import subprocess

RUN_STATUS = 'train_failed'
TRAIN_ERROR = ''
ACTIVE_SUMMARY_EXP_NAME = CFG['experiment_name']

try:
    subprocess.run([
        sys.executable,
        'dogs_vs_cats/src/dinov2_pipeline.py',
        '--mode', 'train',
        '--paths-config', PATHS_CFG,
        '--experiment-config', CFG_PATH,
    ], check=True)
    RUN_STATUS = 'train_ok'
except subprocess.CalledProcessError as exc:
    TRAIN_ERROR = str(exc)
    print('[train] failed, attempting evaluate-only recovery:', exc)
    try:
        subprocess.run([
            sys.executable,
            'dogs_vs_cats/src/dinov2_pipeline.py',
            '--mode', 'evaluate',
            '--paths-config', PATHS_CFG,
            '--experiment-config', CFG_PATH,
            '--summary-suffix', 'train_recovery',
        ], check=True)
        ACTIVE_SUMMARY_EXP_NAME = f"{CFG['experiment_name']}_train_recovery"
        RUN_STATUS = 'train_recovery_eval_ok'
    except subprocess.CalledProcessError as eval_exc:
        TRAIN_ERROR = f"{TRAIN_ERROR} | evaluate_recovery_failed: {eval_exc}"
        RUN_STATUS = 'train_failed'

print('RUN_STATUS:', RUN_STATUS)
print('ACTIVE_SUMMARY_EXP_NAME:', ACTIVE_SUMMARY_EXP_NAME)
if TRAIN_ERROR:
    print('TRAIN_ERROR:', TRAIN_ERROR)


In [ ]:

import json
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

exp_name = ACTIVE_SUMMARY_EXP_NAME if 'ACTIVE_SUMMARY_EXP_NAME' in globals() else CFG['experiment_name']
report_path = Path(PATHS['reports_dir']) / f'{exp_name}_training_summary.json'
summary = {}

if not report_path.exists():
    print('Summary not found, skipping plot review:', report_path)
else:
    summary = json.loads(report_path.read_text())
    print(json.dumps(summary, indent=2))

plot_keys = [
    'train_loss_plot',
    'val_metrics_plot',
    'grad_norm_plot',
    'val_confusion_matrix_plot',
]

if not summary:
    print('No summary payload available to render plots.')
else:
    for key in plot_keys:
        path = summary.get('files', {}).get(key)
        if not path:
            continue
        p = Path(path)
        if not p.exists():
            print('Missing plot file:', p)
            continue
        plt.figure(figsize=(8, 4))
        plt.imshow(mpimg.imread(p))
        plt.title(p.name)
        plt.axis('off')
        plt.show()


In [ ]:

from pathlib import Path

exp_name = ACTIVE_SUMMARY_EXP_NAME if 'ACTIVE_SUMMARY_EXP_NAME' in globals() else CFG['experiment_name']
for section, base_dir in [
    ('metrics', PATHS['metrics_dir']),
    ('predictions', PATHS['predictions_dir']),
    ('plots', PATHS['plots_dir']),
    ('reports', PATHS['reports_dir']),
]:
    print('\\n' + section.upper())
    base = Path(base_dir)
    for item in sorted(base.glob(f'{exp_name}*')):
        print('-', item)


In [ ]:

import sys
import json
import subprocess
from pathlib import Path

base_exp_name = CFG['experiment_name']
base_summary_path = Path(PATHS['reports_dir']) / f'{base_exp_name}_training_summary.json'

if RUN_STATUS != 'train_ok':
    print('Skipping export because training did not complete cleanly. RUN_STATUS=', RUN_STATUS)
elif not base_summary_path.exists():
    print('Skipping export because base training summary is missing:', base_summary_path)
else:
    result = subprocess.check_output([
        sys.executable,
        'dogs_vs_cats/src/export_experiment_artifacts.py',
        '--paths-config', PATHS_CFG,
        '--experiment-config', CFG_PATH,
    ]).decode()
    print(result)
    export_info = json.loads(result)
    manifest_path = Path(export_info['manifest_path'])
    bundle_index_path = Path(export_info['bundle_index_path'])
    print('Exported dir:', export_info['exported_dir'])
    print('Manifest exists:', manifest_path.exists(), manifest_path)
    print('Bundle index exists:', bundle_index_path.exists(), bundle_index_path)
